# Kaggle Dataset Finder: LLM + Semantic Search Pipeline

| | |
|---|---|
| **Type** | Personal project / data-discovery tool |
| **Domain** | NLP · Generative AI · Data engineering |
| **Tools** | Python, Gemini 2.5 Flash, sentence-transformers (all-MiniLM-L6-v2), Kaggle API, pandas, PyTorch |
| **Date** | 2026 |
| **Repository** | [RenzWoo/kaggle-dataset-finder (dummy link)](https://github.com/RenzWoo/kaggle-dataset-finder) |

:::{tip} TL;DR
Type what data you need in plain English and get back a **ranked list of real Kaggle datasets** that were downloaded, opened, and checked. An LLM converts the request into a structured spec, the Kaggle API and sentence embeddings shortlist candidates, and a second LLM pass reads the *actual column names and sample values* to score how well each dataset fits.
:::

<div class="stats-grid">
<div class="stat-card"><div class="stat-num">5</div><div class="stat-desc">pipeline stages, from query to ranked datasets</div></div>
<div class="stat-card"><div class="stat-num">2</div><div class="stat-desc">models: Gemini 2.5 Flash + MiniLM-L6-v2</div></div>
<div class="stat-card"><div class="stat-num">10 → 5</div><div class="stat-desc">search results shortlisted → datasets downloaded</div></div>
<div class="stat-card"><div class="stat-num">0.7 / 0.3</div><div class="stat-desc">weights: requested fields vs. bonus fields</div></div>
</div>

## 1. Problem Statement

Finding a suitable dataset for practice or analysis usually means many keyword searches, opening files, and
checking whether the columns are what you expected. Keyword search on titles also returns off-topic hits.

**Objectives**

- Accept a natural-language request (topic, fields, and intended analysis).
- Retrieve candidates from Kaggle and rank them by semantic similarity, not just keywords.
- **Validate the data itself** (columns and values), not only the dataset description.
- Return an auditable result: relevance score, keyword score, reasoning, use case, and a ready DataFrame.

## 2. Data & Storage

| Stage | Data | How it is stored or managed |
|---|---|---|
| Search metadata | Kaggle dataset `ref`, `title`, `subtitle`, `usability_rating` | Python lists; the search step is memoised with `functools.lru_cache` |
| Embeddings | 384-dimensional MiniLM vectors for the query, titles, and subtitles | PyTorch tensors in memory; the model is loaded once as a singleton (~80 MB weights, cached locally) |
| Downloaded files | Only the **top-N (5)** candidate datasets, unzipped | A `tempfile.TemporaryDirectory` wrapped in a **context manager**, so it is deleted even if a step fails |
| Loaded tables | All `.csv` files found recursively | `dict[str, DataFrame]` keyed by filename; read with `encoding_errors="replace"`; unreadable files are skipped with a warning |
| Results | Ranked matches | List of dicts: `name, keyword_score, relevance_score, reasoning, use_case, dataframe` |
| Secrets | Gemini key, Kaggle token | Environment variables / `.env` (via `python-dotenv`) and `~/.kaggle/kaggle.json`; never hard-coded |

All tunable values live in one `Config` class: usability threshold (0.5), top-K (10), download top-N (5),
keyword pre-filter (0.4), sample values per column (10), and the model names.

## 3. Preprocessing & Data Validation

| Step | What happens | Why it matters |
|---|---|---|
| **Query → schema** | Gemini returns JSON with `topic`, `sub_topic`, `context` (time series, geographic, cross-sectional, panel, list, event log, hierarchical), `desired_fields`, `extra_fields`. Output is cleaned of markdown fences and checked for required keys. | Turns a vague request into fields that can be matched against real columns |
| **Quality filter** | Keep datasets with Kaggle `usability_rating` ≥ 0.5 | Removes poorly documented uploads early |
| **Semantic re-ranking** | Cosine similarity between the topic embedding and both the title and subtitle embeddings; the two scores are averaged; `torch.topk` selects the best | Finds relevant datasets even when wording differs |
| **Selective download** | Only the best 5 are fetched | Saves bandwidth, time, and disk |
| **Column normalisation** | Lower-case and strip spaces, underscores, and hyphens (`Volcano Name` → `volcanoname`), then substring-match against requested fields | Makes field matching robust to naming style |
| **Keyword score** | `0.7 × (desired fields matched) + 0.3 × (extra fields matched)`; tables scoring < 0.4 are dropped before any LLM call | Cheap pre-filter that cuts API cost |
| **Value sampling** | Up to 10 unique non-null values per column are sent to the LLM | Lets the model check that values look right (real dates, real place names) |
| **LLM validation** | Returns `relevance_score` (0–1), one-sentence reasoning that cites observed values, and a concrete use case | Catches datasets whose column names match but whose contents do not |

## 4. Method

```text
Natural-language query
   │
   ▼  [1] Query extraction       Gemini → {topic, sub_topic, context, desired_fields, extra_fields}
   ▼  [2] Search + filtering     Kaggle API → usability ≥ 0.5 → MiniLM cosine re-ranking → top 10
   ▼  [3] Download top-N         5 datasets into an auto-cleaned temp directory
   ▼  [4] Load CSVs              recursive walk → dict of DataFrames
   ▼  [5] Validate + score       keyword pre-filter → Gemini reads columns + sample values
   ▼
Ranked results  (name · relevance_score · keyword_score · reasoning · use_case · dataframe)
```

**Design decisions**

- **Two focused LLM calls** (extraction, then validation) instead of one giant prompt, which keeps each task
  simple and reduces hallucination risk.
- **Cheap filters before expensive calls:** usability filter, embedding re-rank, and keyword pre-filter all run
  before Gemini sees a dataset.
- **Deterministic resource handling:** context manager for temp files, cached model singleton, structured logging.

## 5. Results & Observations

The project has **no formal accuracy benchmark yet**. What the saved runs show:

| Observation | Detail |
|---|---|
| **End-to-end run works** | For a request about *employee information for practice analysis*, the pipeline extracted `topic = "Employee information"` and `context = "cross-sectional"`, found 10 candidates, downloaded 5, loaded 10 CSV files, and returned tables such as an employee master file (ID, department, designation, joining date, salary, employment type). |
| **Raw search is noisy** | For *"population in China by province"*, the top-10 semantic results included a few relevant tables (e.g. China's population by gender and urban/rural) next to off-topic ones (a Mandarin speech corpus, fashion-shopping data). This is why field matching and LLM validation come after retrieval. |
| **Extraction can drop constraints** | In an early prototype, that same request was parsed as a *time series* with fields `[year, population]`, silently dropping **province**. The final prompt adds few-shot examples and explicit inference rules, which is the right place to re-test. |
| **API limits are real** | One saved run hit a Gemini quota error (HTTP 429), highlighting the need for retry and back-off logic. |

**Key takeaway:** semantic search gets you a shortlist; **inspecting the real columns and values** is what
separates a dataset that *sounds* right from one that *is* right.

## 6. Data Science Concepts Demonstrated

| Concept | Where it appears |
|---|---|
| **Text embeddings & semantic search** | MiniLM sentence embeddings + cosine similarity to re-rank titles and subtitles |
| **LLM structured extraction** | Prompted JSON schema with few-shot examples and validation of required keys |
| **Data validation** | Column-level matching and value-level checks before trusting a dataset |
| **API integration** | Kaggle API (search, download) and Gemini API (extraction, validation) |
| **Resource & pipeline design** | Temp-directory context manager, `lru_cache`, singleton models, centralised config |
| **Scoring & thresholds** | Weighted keyword score, pre-filter threshold, LLM relevance score |
| **Data wrangling** | Recursive file discovery, encoding-tolerant CSV loading, robust error handling |
| **Secrets management** | Environment variables and `.env` instead of hard-coded keys |

## 7. Limitations & Next Steps

- **Add an evaluation set.** Hand-label relevance for ~30 queries and report precision@k / nDCG, comparing
  keyword-only, embedding-only, and the full pipeline.
- **Validate before downloading.** Use Kaggle file listings and metadata (or partial reads) to inspect columns
  first, so whole datasets are not downloaded just to be rejected.
- **Smarter field matching.** Replace substring matching with embedding-based column matching (e.g. `pop`
  ≈ `population`, `prov` ≈ `province`).
- **Handle more formats.** Currently CSV only; add Excel, JSON, and Parquet.
- **Resilience.** Retry with exponential back-off for rate limits; cache LLM responses per query.
- **Code hygiene.** Fix a malformed log format string, add unit tests for the scoring functions, and package
  the tool as a CLI or a small Streamlit app.
- **Licensing and provenance.** Surface dataset licences and last-updated dates in the results.